In [ ]:
# Verificar GPU disponible
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("ADVERTENCIA: No se detectó GPU. El entrenamiento será muy lento.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
from transformers import BertForMaskedLM, BertTokenizer, AutoTokenizer, DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset
import numpy as np

In [ ]:
# Solo ejecutar si no están instaladas las dependencias
%pip install evaluate transformers datasets torch pandas scikit-learn --quiet

#### Import and split

- Uses Pandas DataFrames to split the data into train, validation and test using train_test_split() from sklearn and its stratify parameter for preserving class proportions.

- Converts DataFrames to Datasets and renames columns to aling data structure to be able to use Hugging Face's pre-train model.

In [ ]:
# Cargar dataset (ruta local)
df_input = pd.read_csv('train.csv')
print(f"Dataset total: {len(df_input):,} registros")

# Filtrar solo español
df_input = df_input[df_input['language'] == 'spanish']
print(f"Solo español: {len(df_input):,} registros")

# DATASET COMPLETO SIN FILTROS DE CATEGORÍAS
df_input_full = df_input.drop(["language", "label_quality"], axis=1)

print(f"Total categorías: {df_input_full['category'].nunique()}")

# Split estratificado: 90% train, 5% val, 5% test
train_df, temp_df = train_test_split(
    df_input_full, 
    test_size=0.1,  # 10% para val+test
    random_state=42, 
    stratify=df_input_full['category']
)

val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5,  # 50% de temp = 5% del total
    random_state=42, 
    stratify=temp_df['category']
)

# Convertir a Datasets de HuggingFace
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Renombrar columnas
train_dataset = train_dataset.rename_column("title", "text").rename_column("category", "labels")
val_dataset = val_dataset.rename_column("title", "text").rename_column("category", "labels")
test_dataset = test_dataset.rename_column("title", "text").rename_column("category", "labels")

print(f"\nSplit final:")
print(f"  Train: {len(train_dataset):,}")
print(f"  Val:   {len(val_dataset):,}")
print(f"  Test:  {len(test_dataset):,}")

#### Preprocess

- Load a BETO cased tokenizer to preprocess the text field.
- Tokenize, pad, and truncate for training.
- Create a map of the expected ids to their labels.

In [ ]:
# Create a mapping from category names to ids
label_to_id = {label: id for id, label in enumerate(train_dataset.unique('labels'))}
id_to_label = {id: label for label, id in label_to_id.items()}
print(f"Total labels: {len(label_to_id)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'dccuchile/bert-base-spanish-wwm-cased',
    num_labels = len(train_dataset.unique('labels')),
    id2label = id_to_label,
    label2id = label_to_id)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1) # TODO: axis = -1 ?
    # Use the mapped labels for computation
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
# Map string labels to integers
def map_labels_to_ids(examples):
    return {'labels': label_to_id[examples['labels']]}

print("Tokenizando train...")
train_dataset = train_dataset.map(preprocess_function, batched=True, batch_size=1000)
print("Tokenizando val...")
val_dataset = val_dataset.map(preprocess_function, batched=True, batch_size=1000)
print("Tokenizando test...")
test_dataset = test_dataset.map(preprocess_function, batched=True, batch_size=1000)

print("Mapeando labels...")
train_dataset = train_dataset.map(map_labels_to_ids)
val_dataset = val_dataset.map(map_labels_to_ids)
test_dataset = test_dataset.map(map_labels_to_ids)

print("Preprocesamiento completo!")

In [11]:
# Display the first tokenized example from the train_dataset
print(train_dataset[0])

{'text': 'Hidrolavadora Industrial Autónoma (explosión) 200bar 15l/m', 'labels': 0, 'input_ids': [4, 29208, 23700, 13794, 15314, 16152, 1147, 11315, 1135, 1272, 1757, 1860, 30938, 972, 1027, 5], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


#### Create Model and Train

- Define your training hyperparameters in TrainingArguments. At the end of each epoch, the Trainer will evaluate the accuracy and save the training checkpoint.

- Pass the training arguments to Trainer along with the model, dataset and compute_metrics function.

- TODO: Use datacollator().

  - WHY IS TOKENIZER NOT USED in this example? How does it work within the Training?

- Call train() to finetune your model.

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="beto_model_full_es",
    
    # Learning rate
    learning_rate=2e-5,
    
    # Batch sizes optimizados para RTX 3060 12GB
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    
    # Gradient accumulation (efectivamente batch=32)
    gradient_accumulation_steps=2,
    
    # Epochs
    num_train_epochs=2,
    
    # Regularización
    weight_decay=0.01,
    warmup_ratio=0.1,  # 10% warmup
    
    # Evaluación y guardado
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    
    # OPTIMIZACIONES GPU - RTX 3060
    fp16=True,  # Mixed precision - CRÍTICO para ahorrar VRAM
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    
    # Logging
    logging_dir="./logs",
    logging_steps=500,
    report_to="none",  # Desactivar wandb
    
    # Checkpoints
    save_total_limit=2,
    
    # Misc
    push_to_hub=False,
    seed=42,
)

# Callback para early stopping (opcional pero recomendado)
early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

print("Iniciando entrenamiento...")
print(f"Steps por epoch: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps):,}")

trainer.train()

In [ ]:
trainer.save_model("beto_model_full_es")
print("Modelo guardado en: beto_model_full_es/")

In [ ]:
print("Evaluando en test set...")
results = trainer.evaluate(test_dataset)
print(f"\nResultados en Test:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

#### Use the model with test dataset

predictions contains the logic (similar to probability) of each category. To get the most likely category np.argmax(predictions.predictions, axis=-1)

In [ ]:
# Predicciones en test
predictions = trainer.predict(test_dataset)

predicted_ids = np.argmax(predictions.predictions, axis=-1)
predicted_labels = [id_to_label[i] for i in predicted_ids]

# Mostrar ejemplos
num_examples = 10
print(f"\n{'='*60}")
print(f"Ejemplos de predicciones ({num_examples} muestras):")
print(f"{'='*60}")

for i in range(num_examples):
    true_label = id_to_label[test_dataset['labels'][i]]
    pred_label = predicted_labels[i]
    match = "✓" if true_label == pred_label else "✗"
    
    print(f"\n{match} Texto: {test_dataset['text'][i][:80]}...")
    print(f"   Real: {true_label}")
    print(f"   Pred: {pred_label}")

#### Metricas adicionales

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Calcular métricas
true_labels = [id_to_label[test_dataset['labels'][i]] for i in range(len(test_dataset))]

print(f"\n{'='*60}")
print("MÉTRICAS FINALES")
print(f"{'='*60}")
print(f"Accuracy: {accuracy_score(true_labels, predicted_labels):.4f}")
print(f"F1 Macro: {f1_score(true_labels, predicted_labels, average='macro'):.4f}")
print(f"F1 Weighted: {f1_score(true_labels, predicted_labels, average='weighted'):.4f}")

# Classification report (top categorías)
print(f"\n{'='*60}")
print("Classification Report (primeras 20 categorías):")
print(f"{'='*60}")
print(classification_report(true_labels, predicted_labels, zero_division=0))